[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/python-ai-business-data-science/blob/main/fast_track/13_notebook_to_project.ipynb)

# 📓 Notebook 13 (fast track) — From Notebook to Project

> **Module:** Production · **Estimated time:** 50–70 min · **Difficulty:** Intermediate

A notebook is a great place to *figure things out*. It is a terrible place to *deploy* things from. The moment your code matters — someone else uses it, you run it on a schedule, it touches a database — you need to graduate from `.ipynb` to a real Python project.

This notebook walks through that transition. We take the toolkit you've already written (`call_cost`, `MockLLM`, `extract_invoice`) and turn it into a proper package with:

- A standard folder structure.
- A `pyproject.toml` that declares dependencies.
- Real **pytest** tests.
- A small **command-line interface** (`costkit total ...`).
- A reproducible **virtualenv** workflow.

We do it *inside this notebook* using the filesystem — creating the files, importing the package, running pytest on it — so you can see every step actually work.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Lay out a Python project with the standard `src/` layout.
2. Write a `pyproject.toml` that declares your package and its dependencies.
3. Create a **virtualenv** and `pip install` your project in editable mode.
4. Write **pytest** tests with fixtures and parametrize.
5. Build a **CLI** with `argparse` (or `typer`).
6. Run **type checks** with mypy and **linting** with ruff.
7. Decide what to put in `__init__.py`, what to expose, what to keep private.

## ✅ Prerequisites

Notebook 5 (functions & modules) and NB 6 (classes); ideally NB 17 / NB 20 (the `MockLLM`, `call_cost`, `extract_invoice` code we package here).

> 🏎️ **You're on the fast track.** This is a trimmed version of the canonical [`06_production/22_from_notebook_to_project.ipynb`](../06_production/22_from_notebook_to_project.ipynb) — from notebook to a real project. The Stretch exercises (A–D) and the 🎁 Bonus mini-project have been removed to keep the notebook lean. Open the canonical version once you want the deeper material.

---


## 1. Why graduate from a notebook?

Notebooks are perfect for *exploration*. They are bad at:

| Need | Why notebooks fail | What a project gives you |
|---|---|---|
| **Reuse** | Hard to import notebook code from another notebook. | `from costkit import call_cost` — works anywhere. |
| **Versioning** | Notebooks are JSON files with embedded outputs; diffs are unreadable. | Plain `.py` files diff cleanly. |
| **Testing** | No good way to run automated tests against a notebook. | `pytest` lives natively in a project. |
| **CI/CD** | Hard to run notebooks on every PR. | A project plugs into GitHub Actions in 5 lines. |
| **Deployment** | A scheduler can't `pip install` a notebook. | A package gets a version number and installs anywhere. |

> 🎯 **The rule of thumb.** Once *two* notebooks copy-paste the same function, move that function into a project. Once *one* notebook needs to be re-run on a schedule, build the project around it.

## 2. The standard layout — `src/` vs flat

For 90% of projects, this is the layout you want:

```
my-project/
├── pyproject.toml            ← project metadata + dependencies
├── README.md
├── .gitignore
├── src/
│   └── costkit/              ← the importable package
│       ├── __init__.py        ← public API
│       ├── cost.py            ← module: cost calculations
│       └── parsing.py         ← module: defensive parsers
├── tests/
│   ├── test_cost.py
│   └── test_parsing.py
└── scripts/
    └── monthly_report.py     ← entry point a scheduler can run
```

The `src/` layout (instead of putting `costkit/` directly at the root) is the modern best practice: it prevents accidentally importing your code from the current directory instead of from the installed package.

## 3. Build the project — for real, in `/tmp`

To make this concrete, we'll **actually create** all those files in a temp directory and inspect the result.

In [ ]:
import os
import shutil
import sys
import subprocess
from pathlib import Path
from textwrap import dedent

# Use a writable temp directory so this notebook works on any system
PROJECT_ROOT = Path("/tmp/costkit_demo")
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

# Folder structure
(PROJECT_ROOT / "src" / "costkit").mkdir(parents=True)
(PROJECT_ROOT / "tests").mkdir()

print(f"Project root: {PROJECT_ROOT}")
print("Folders created:")
for p in sorted(PROJECT_ROOT.rglob("*")):
    print(f"  {p.relative_to(PROJECT_ROOT)}/")


## 4. Write the package modules

In [ ]:
# src/costkit/cost.py — a small set of pure functions
COST_PY = '''
"""Token-cost calculations for LLM workloads."""

from typing import Union

Number = Union[int, float]

DEFAULT_PRICE_IN_PER_1K  = 0.0006
DEFAULT_PRICE_OUT_PER_1K = 0.0024


def call_cost(tokens_in: int, tokens_out: int,
              price_in_per_1k:  Number = DEFAULT_PRICE_IN_PER_1K,
              price_out_per_1k: Number = DEFAULT_PRICE_OUT_PER_1K) -> float:
    """USD cost of one LLM call given input/output token counts."""
    if tokens_in < 0 or tokens_out < 0:
        raise ValueError("token counts must be non-negative")
    return (tokens_in  / 1000 * price_in_per_1k +
            tokens_out / 1000 * price_out_per_1k)


def batch_cost(records: list[dict]) -> float:
    """Total cost across a batch of records, each with tokens_in/out."""
    return sum(call_cost(r["tokens_in"], r["tokens_out"]) for r in records)
'''

(PROJECT_ROOT / "src" / "costkit" / "cost.py").write_text(COST_PY)
print("✓ src/costkit/cost.py")


In [ ]:
# src/costkit/parsing.py — defensive parsers we used everywhere
PARSING_PY = '''
"""Defensive parsers for messy real-world input."""

from typing import Any, Optional


def safe_get(d: Any, path: list) -> Any:
    """Walk a nested dict/list by a path; return None on any missing step."""
    cur = d
    for step in path:
        try:
            cur = cur[step]
        except (KeyError, IndexError, TypeError):
            return None
    return cur


def parse_number(text: Any) -> Optional[float]:
    """Try to convert *text* to float; return None when it cannot."""
    try:
        return float(text)
    except (ValueError, TypeError):
        return None
'''

(PROJECT_ROOT / "src" / "costkit" / "parsing.py").write_text(PARSING_PY)
print("✓ src/costkit/parsing.py")


In [ ]:
# src/costkit/__init__.py — the public API of the package
INIT_PY = '''
"""costkit — small helpers for LLM cost and parsing."""
from .cost    import call_cost, batch_cost
from .parsing import safe_get, parse_number

__all__   = ["call_cost", "batch_cost", "safe_get", "parse_number"]
__version__ = "0.1.0"
'''

(PROJECT_ROOT / "src" / "costkit" / "__init__.py").write_text(INIT_PY)
print("✓ src/costkit/__init__.py")


**Why `__init__.py`?** It marks the folder as a package and decides what's importable. The lines above mean a user can write `from costkit import call_cost` instead of `from costkit.cost import call_cost`. Keep this file small — it's your package's **public API**.

## 5. The project metadata — `pyproject.toml`

`pyproject.toml` is the modern standard for declaring a Python project — replacing the older `setup.py`/`setup.cfg` files. A minimal but complete `pyproject.toml`:

In [ ]:
PYPROJECT = '''
[build-system]
requires      = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name            = "costkit"
version         = "0.1.0"
description     = "Small helpers for LLM cost and parsing."
readme          = "README.md"
requires-python = ">=3.10"
license         = {text = "MIT"}
authors         = [{name = "Course materials"}]

# Runtime dependencies (only what costkit needs)
dependencies = [
    # (no extra deps for this tiny package)
]

# Optional development dependencies — installed with `pip install -e ".[dev]"`
[project.optional-dependencies]
dev = ["pytest>=8.0", "mypy>=1.8", "ruff>=0.4"]

# CLI entry points — `costkit ...` becomes runnable after install
[project.scripts]
costkit = "costkit.cli:main"

# Configure the src/ layout for setuptools
[tool.setuptools.packages.find]
where = ["src"]
'''

(PROJECT_ROOT / "pyproject.toml").write_text(PYPROJECT)
print("✓ pyproject.toml")


**Things to notice in this file:**

- **`name = "costkit"`** — pip-installable as `pip install costkit` (once you publish).
- **`requires-python = ">=3.10"`** — fail loudly on older Python.
- **`[project.optional-dependencies]`** — `pip install -e ".[dev]"` adds dev tools; production installs stay lean.
- **`[project.scripts]`** — declares a command-line entry point. After install, `costkit total ...` works in any shell.

## 6. The CLI

A command-line interface is what lets a scheduler (cron, Airflow, GitHub Actions) actually run your code. Use the standard-library `argparse` for simple CLIs, or `typer` / `click` for nicer ones.

In [ ]:
CLI_PY = '''
"""costkit command-line interface."""
import argparse
import sys
from . import call_cost, batch_cost

def main(argv=None):
    parser = argparse.ArgumentParser(prog="costkit",
                                      description="LLM cost calculator")
    sub = parser.add_subparsers(dest="cmd", required=True)

    # `costkit total <tokens_in> <tokens_out>`
    p = sub.add_parser("total", help="Cost of one call")
    p.add_argument("tokens_in",  type=int)
    p.add_argument("tokens_out", type=int)
    p.add_argument("--in-price",  type=float, default=0.0006)
    p.add_argument("--out-price", type=float, default=0.0024)

    args = parser.parse_args(argv)
    if args.cmd == "total":
        cost = call_cost(args.tokens_in, args.tokens_out,
                          args.in_price, args.out_price)
        print(f"${cost:.5f}")
        return 0
    return 1


if __name__ == "__main__":
    sys.exit(main())
'''

(PROJECT_ROOT / "src" / "costkit" / "cli.py").write_text(CLI_PY)
print("✓ src/costkit/cli.py")


## 7. The tests

`pytest` is the de-facto standard testing framework for Python. Tests live in their own folder so you don't ship them inside the package.

In [ ]:
TEST_COST_PY = '''
import pytest
from costkit import call_cost, batch_cost


def test_zero_tokens_zero_cost():
    assert call_cost(0, 0) == 0


def test_known_value():
    # 1000 in @ $0.0006 + 500 out @ $0.0024 = 0.0006 + 0.0012 = 0.0018
    assert call_cost(1000, 500) == pytest.approx(0.0018, rel=1e-6)


def test_custom_pricing():
    assert call_cost(1000, 0, price_in_per_1k=0.001) == pytest.approx(0.001)


def test_negative_tokens_raise():
    with pytest.raises(ValueError):
        call_cost(-1, 0)


@pytest.mark.parametrize("records, expected", [
    ([],                                                       0.0),
    ([{"tokens_in": 1000, "tokens_out": 0}],                   0.0006),
    ([{"tokens_in": 1000, "tokens_out": 0}] * 3,               0.0018),
])
def test_batch_cost(records, expected):
    assert batch_cost(records) == pytest.approx(expected, rel=1e-6)
'''

(PROJECT_ROOT / "tests" / "test_cost.py").write_text(TEST_COST_PY)
print("✓ tests/test_cost.py")


In [ ]:
TEST_PARSING_PY = '''
from costkit import safe_get, parse_number


def test_safe_get_basic():
    d = {"a": {"b": {"c": 42}}}
    assert safe_get(d, ["a", "b", "c"]) == 42


def test_safe_get_missing_returns_none():
    d = {"a": {"b": {}}}
    assert safe_get(d, ["a", "b", "c"]) is None


def test_safe_get_index_error_returns_none():
    d = {"items": [1, 2]}
    assert safe_get(d, ["items", 99]) is None


def test_parse_number_ok():
    assert parse_number("1.5") == 1.5
    assert parse_number(2)    == 2.0


def test_parse_number_bad():
    assert parse_number("nope") is None
    assert parse_number(None)   is None
'''

(PROJECT_ROOT / "tests" / "test_parsing.py").write_text(TEST_PARSING_PY)
print("✓ tests/test_parsing.py")


## 8. README + .gitignore

The two files every project needs.

In [ ]:
README = '''
# costkit

Small helpers for LLM cost arithmetic and defensive JSON parsing.

## Install

```bash
pip install -e ".[dev]"
```

## Use

```python
from costkit import call_cost

call_cost(1000, 500)    # → 0.0018
```

## CLI

```bash
costkit total 1000 500
```
'''
(PROJECT_ROOT / "README.md").write_text(README)

GITIGNORE = '''
__pycache__/
*.pyc
.pytest_cache/
.mypy_cache/
.ruff_cache/
.venv/
build/
dist/
*.egg-info/
'''
(PROJECT_ROOT / ".gitignore").write_text(GITIGNORE)

print("✓ README.md")
print("✓ .gitignore")


## 9. Install the package and run the tests

Let's actually install the package (using `pip install -e`) and run the test suite — *from inside this notebook*.

In [ ]:
# `pip install -e ".[dev]"` would install in editable mode; here we shortcut
# by adding src/ to sys.path so we can `import costkit` directly.
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Verify it imports
import costkit
print(f"costkit version : {costkit.__version__}")
print(f"costkit location: {costkit.__file__}")

# Try the public API
print(f"\ncall_cost(1000, 500) = ${costkit.call_cost(1000, 500):.5f}")
print(f"safe_get({{'a': {{'b': 42}}}}, ['a','b'])  = {costkit.safe_get({'a': {'b': 42}}, ['a', 'b'])}")


In [ ]:
# Run the tests using pytest's API (works from inside a notebook)
import pytest as pt

# pytest -q  --tb=short  /tmp/costkit_demo/tests
result = pt.main(["-q", "--tb=short", "--no-header",
                   str(PROJECT_ROOT / "tests")])
print(f"\nExit code: {result}   (0 = all tests passed)")


> 🎯 **What just happened.** You wrote a real Python package, installed it (in spirit), and ran a real test suite — all from a notebook. The same `pytest` command runs in CI, in your terminal, in your IDE's test runner. The pattern is portable.

## 10. Type hints and linting — the next layer

Two tools every modern Python project uses. They are **optional** in the sense that your code runs without them, but **mandatory** in the sense that every serious codebase has them configured.

### `mypy` — static type checking

Catches `TypeError`s before you run the code. Configure with `[tool.mypy]` in `pyproject.toml`:

```toml
[tool.mypy]
strict       = true
ignore_missing_imports = true
```

Run with `mypy src/`. With strict mode on, missing type hints become errors — which forces good habits.

### `ruff` — linting + autoformatting

A fast linter that catches dead imports, unused variables, style issues, common bugs. Configure in `pyproject.toml`:

```toml
[tool.ruff]
line-length = 100
target-version = "py310"
[tool.ruff.lint]
select = ["E", "F", "W", "I", "UP"]   # pyflakes, pep8, isort, pyupgrade
```

Run with `ruff check src/ tests/` and `ruff format src/ tests/`. It's *fast* (full-codebase lint in ~50 ms) and replaces a half-dozen older tools (flake8, isort, black-ish formatting).

## 11. The virtualenv workflow — for completeness

Every project should live in its own **virtual environment** so its dependencies don't collide with another project's.

```bash
# Create
python -m venv .venv
source .venv/bin/activate     # macOS / Linux
.venv\Scripts\activate        # Windows

# Install your project in editable mode (changes to src/ take effect immediately)
pip install -e ".[dev]"

# Run things
pytest
mypy src/
ruff check src/ tests/
costkit total 1000 500        # the CLI we wired up earlier
```

> 💡 **`uv`** (`pip install uv` once) is a newer, faster replacement for the `pip` + `venv` toolchain. The workflow is the same; `uv venv` / `uv pip install -e ".[dev]"` just runs in a second instead of thirty.

## 12. Where to go next — the rest of "production Python"

Things that are out of scope for this notebook but become useful as the project grows:

| Topic | What it does | Tool |
|---|---|---|
| **Pre-commit hooks** | Auto-format and lint on every commit | `pre-commit` |
| **CI (GitHub Actions)** | Run tests on every PR | `.github/workflows/ci.yml` |
| **Versioning** | Automatic version bumps from git tags | `setuptools-scm` |
| **Building wheels** | `dist/costkit-0.1.0-py3-none-any.whl` | `python -m build` |
| **Publishing** | Push the wheel to PyPI | `twine upload` |
| **Documentation** | API docs from docstrings | `mkdocs` + `mkdocstrings` |
| **Coverage** | Which lines do tests exercise? | `pytest --cov` (via `coverage.py`) |

You don't need all of these to ship. You *do* need a `pyproject.toml`, a test suite, and a CLI — which is what we just built.

## 🧪 Practice exercises

### Exercise 1 — ⭐ Add a new module

Create `src/costkit/cleaning.py` with one function `clean_latencies(values, max_ms=60000)` that filters out non-numeric and out-of-range latencies (reuse the version from NB 5). Add it to `__init__.py` so it's importable as `from costkit import clean_latencies`.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
CLEANING_PY = '''
"""Defensive cleaners for messy time-series and log data."""
from .parsing import parse_number

def clean_latencies(values, max_ms=60_000):
    out = []
    for v in values:
        n = parse_number(v) if isinstance(v, str) else v
        if isinstance(n, (int, float)) and 0 <= n <= max_ms:
            out.append(float(n))
    return out
'''
(PROJECT_ROOT / "src" / "costkit" / "cleaning.py").write_text(CLEANING_PY)

# Append to __init__.py
init = (PROJECT_ROOT / "src" / "costkit" / "__init__.py").read_text()
init = init.replace(
    "from .parsing import safe_get, parse_number",
    "from .parsing import safe_get, parse_number\nfrom .cleaning import clean_latencies",
).replace(
    '"parse_number"',
    '"parse_number", "clean_latencies"',
)
(PROJECT_ROOT / "src" / "costkit" / "__init__.py").write_text(init)

# Reload and verify
import importlib, costkit; importlib.reload(costkit)
print(costkit.clean_latencies(["120", "bad", -5, 70_000, 800]))   # → [120.0, 800.0]
```

**Lesson.** Adding a feature to a package = create module → import from `__init__.py` → write a test. That third step is the one most people skip; don't.
</details>

### Exercise 2 — ⭐⭐ Test the new module

Write `tests/test_cleaning.py` that covers the new function: a happy path, a string mixed in, a too-large value, an out-of-range negative. Run the suite again and confirm 4 more tests pass.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
TEST_CLEANING_PY = '''
import pytest
from costkit import clean_latencies


def test_happy_path():
    assert clean_latencies([100, 200, 300]) == [100.0, 200.0, 300.0]

def test_strings_parsed():
    assert clean_latencies(["120", "bad", 800]) == [120.0, 800.0]

def test_too_large_filtered():
    assert clean_latencies([100, 70_000], max_ms=60_000) == [100.0]

def test_negative_filtered():
    assert clean_latencies([-5, 100]) == [100.0]
'''
(PROJECT_ROOT / "tests" / "test_cleaning.py").write_text(TEST_CLEANING_PY)

import pytest as pt
pt.main(["-q", "--tb=short", str(PROJECT_ROOT / "tests")])
```

You should see the test count jump by four; all green. **Always commit code + test together.**
</details>

### Exercise 3 — ⭐⭐ Pre-flight runtime check

Add a tiny script `scripts/sanity_check.py` that imports the package, calls one function, and exits with status code `0` on success or `1` on failure. A scheduler can run this every minute to verify the package is healthy.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
SANITY_PY = '''
#!/usr/bin/env python
"""Quick post-deploy sanity check — should run in well under 1 second."""
import sys
try:
    from costkit import call_cost, safe_get
    assert call_cost(1000, 500) > 0
    assert safe_get({"a": {"b": 1}}, ["a", "b"]) == 1
    print("OK")
    sys.exit(0)
except Exception as e:
    print(f"FAIL: {type(e).__name__}: {e}")
    sys.exit(1)
'''
(PROJECT_ROOT / "scripts").mkdir(exist_ok=True)
(PROJECT_ROOT / "scripts" / "sanity_check.py").write_text(SANITY_PY)

# Verify it works
import subprocess, sys
env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT / "src")}
proc = subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "sanity_check.py")],
                      capture_output=True, text=True, env=env)
print(f"stdout: {proc.stdout.strip()}")
print(f"exit  : {proc.returncode}")
```

**Why this matters.** A 1-second healthcheck after every deploy is the cheapest way to catch *everything-just-broke* before users notice. It is the difference between "fail at 3 am" and "fail before you log off".
</details>

### Exercise 4 — ⭐⭐ Debug me 🐞

You've installed the package in editable mode and the tests pass. But importing it fails with `ImportError: cannot import name 'call_cost' from 'costkit'`. The source clearly has the function. What's the likely cause?

<details>
<summary>💡 <b>Solution</b></summary>

Two common causes, both easy to miss:

1. **`call_cost` is defined but not exported from `__init__.py`.** The package only exposes what its `__init__.py` imports. Add `from .cost import call_cost`.
2. **You have a second copy of the package** earlier on `sys.path` (e.g. an old global-install, or a stray `costkit/` folder in the project root). Run `python -c "import costkit; print(costkit.__file__)"` to see which one is loaded.

**Lesson.** When imports do weird things, *always* check `__file__` first. The error message says *what* it can't find; the file path tells you *where it's looking*.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise C — ⭐⭐⭐ Add a CLI entry point

Extend the `costkit` package from this notebook with a tiny CLI: invoking `python -m costkit 1500 750` should print the expected cost given 1,500 input tokens and 750 output tokens at the default per-1K prices.

Sketch the `__main__.py` file's contents and demonstrate it by running the module via `subprocess`.

In [ ]:
# Your code here  👇
import subprocess, sys, os
from pathlib import Path

PROJECT = Path("/tmp/costkit_demo")
# Assume the package skeleton was already created by an earlier cell. We add __main__.py.
main_py = PROJECT / "src" / "costkit" / "__main__.py"
# Write the file, then run it.
# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import subprocess, sys, os
from pathlib import Path

PROJECT = Path("/tmp/costkit_demo")
main_py = PROJECT / "src" / "costkit" / "__main__.py"
main_py.write_text('''
import sys
from .cost import call_cost

def main(argv=None):
    argv = argv if argv is not None else sys.argv[1:]
    if len(argv) != 2:
        print("usage: python -m costkit TOKENS_IN TOKENS_OUT")
        return 2
    tokens_in, tokens_out = map(int, argv)
    print(f"${call_cost(tokens_in, tokens_out):.4f}")
    return 0

if __name__ == "__main__":
    raise SystemExit(main())
''')

env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT / "src") + os.pathsep + env.get("PYTHONPATH", "")
out = subprocess.run([sys.executable, "-m", "costkit", "1500", "750"],
                     capture_output=True, text=True, env=env, cwd=str(PROJECT))
print("stdout:", out.stdout.strip())
print("returncode:", out.returncode)
```

**Reasoning.** Three things `__main__.py` lets you do for free. (1) `python -m costkit ARGS` is the standard way to invoke a package as a script — no `setup.py entry_points` needed. (2) A `main(argv=None)` function (that takes an argv list, defaulting to `sys.argv[1:]`) is **testable**: a unit test can pass synthetic argv without involving a real subprocess. (3) `raise SystemExit(main())` (vs `sys.exit(main())`) is slightly safer when `main` may be imported by other code — it's the conventional shape in production CLIs.
</details>

### Stretch exercise D — ⭐⭐⭐ Write a property-based test

Pytest property-based tests verify *invariants* across many auto-generated inputs. Without installing `hypothesis`, write a hand-rolled property test that:

- Generates 100 random `(tokens_in, tokens_out)` pairs in `[0, 10000]`.
- Asserts `call_cost` is **monotonic in each argument** (more tokens → ≥ cost).
- Asserts the cost is **always non-negative**.

Use the `call_cost` defined in your `costkit` package.

In [ ]:
# Your code here  👇
import random

def call_cost(tokens_in, tokens_out,
              price_in=0.0006, price_out=0.0024):
    return (tokens_in/1000)*price_in + (tokens_out/1000)*price_out

# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import random

def call_cost(tokens_in, tokens_out,
              price_in=0.0006, price_out=0.0024):
    return (tokens_in/1000)*price_in + (tokens_out/1000)*price_out

rng = random.Random(0)
for _ in range(100):
    a = rng.randint(0, 10_000)
    b = rng.randint(0, 10_000)
    base = call_cost(a, b)
    assert base >= 0, f"cost went negative for a={a}, b={b}"
    assert call_cost(a + 1, b)     >= base, "monotone in tokens_in"
    assert call_cost(a,     b + 1) >= base, "monotone in tokens_out"

print("100 property checks passed ✅")
```

**Reasoning.** Property-based tests catch a different class of bugs from example tests. Three ideas. (1) **Invariants** are the assertions that should be true *for every input*, not just for the three you happened to write down. Monotonicity in this case rules out a pricing bug where someone accidentally subtracts. (2) **Seeded randomness** keeps the test reproducible — the assert message tells you exactly which input failed. (3) For real projects use `hypothesis` — it shrinks failing inputs to the minimum reproducer and integrates with pytest. Hand-rolling like this is fine for sketches and is enough to plant the habit.
</details>

## 🧠 Key takeaways

1. **Two notebooks → one project.** As soon as you copy-paste a function, package it.
2. The standard layout is `src/your_package/` + `tests/` + `pyproject.toml` + `README.md`.
3. **`__init__.py` is the public API.** Be deliberate about what you export.
4. **`pyproject.toml`** declares everything: name, version, deps, optional dev deps, CLI entry points.
5. **`pytest`** is the default test runner; tests are plain `.py` files under `tests/`.
6. **`mypy` and `ruff`** add static safety and consistency — install them as `[project.optional-dependencies] dev`.
7. **Virtualenvs are non-negotiable.** Use `python -m venv` or `uv`.
8. A CLI entry point (`[project.scripts]`) is what lets a scheduler actually run your code.

## ✅ Self-assessment

- [ ] Lay out a project with `src/`, `tests/`, `pyproject.toml`
- [ ] Write a `pyproject.toml` with deps and a CLI entry point
- [ ] Add `pytest` tests including parametrize and an exception check
- [ ] Add a function, add it to `__init__.py`, write a test for it
- [ ] Run `pytest` and read its output
- [ ] Explain when to import from `costkit.cost` vs from `costkit`

## 🎉 That's the fast track

You've gone from `print("Hello")` all the way to fetching real data, training models, building LLM workflows and agents, and packaging it into a proper project — the whole pipeline, end to end.

**Where to go from here:** open the full course at [`../00_onboarding/00_master_onboarding.ipynb`](../00_onboarding/00_master_onboarding.ipynb) for the deep versions — interview-grade Stretch exercises in every notebook, plus NumPy, time-series forecasting, model evaluation, feature engineering, document processing, AI observability, scheduling, the two capstones, and the industry-application modules.